# Market Data Aggregation & Paper Trading API - Demo Client

**M2EIF 2025/2026 - Université Paris-Dauphine**

This notebook demonstrates all API features:
- Authentication (registration, login)
- Market Data (best_touch, trades, klines, EWMA)
- Fund Management (deposits, balances)
- Order Management (limit, market, IOC)
- Order Modification (bonus)
- Order Cancellation
- WebSocket Order Management (bonus)

## Setup & Configuration

## Start API Server 

The following cell will:
1. Generate a random SECRET_KEY (required for JWT authentication)
2. Start the API server in the background
3. Wait for it to be ready

**Run it before executing any other demo cells.**

In [17]:
import os
import secrets
import subprocess
import time
import requests

# Generate random SECRET_KEY (32 bytes = 64 hex chars)
SECRET_KEY = secrets.token_hex(32)
os.environ["SECRET_KEY"] = SECRET_KEY

print("Generated SECRET_KEY:", SECRET_KEY[:16] + "..." + SECRET_KEY[-16:])

# Start the API server in background
print("Starting API server...")
server_process = subprocess.Popen(
    ["python", "run_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=os.environ.copy()
)

# Wait for server to be ready (max 10 seconds)
print("Waiting for server to start...")
for i in range(20):
    try:
        response = requests.get("http://localhost:8000/docs", timeout=1)
        if response.status_code == 200:
            print(f"Server ready at http://localhost:8000")
            print(f"API docs: http://localhost:8000/docs")
            break
    except:
        time.sleep(0.5)
else:
    print("Server might not be ready yet, but continuing...")

print("\n✓ You can now run the demo cells below!")

Generated SECRET_KEY: e31cfca2ff9c0e60...61015c3897960fcd
Starting API server...
Waiting for server to start...
Server ready at http://localhost:8000
API docs: http://localhost:8000/docs

✓ You can now run the demo cells below!


In [18]:
import requests
import json
import time
import asyncio
import websockets

# Configuration
BASE_URL = "http://localhost:8000"
WS_URL = "ws://localhost:8000/ws"
USERNAME = f"demo_user_{int(time.time())}"
PASSWORD = "SecurePass123!"
TOKEN = None
HEADERS = {}

In [19]:
def print_section(title):
    print("\n" + "=" * 80)
    print(f"  {title}")
    print("=" * 80 + "\n")

def print_subsection(title):
    print(f"\n--- {title} ---\n")

## 1. Authentication

In [20]:
def test_authentication():
    global TOKEN, HEADERS
    
    print_section("1. AUTHENTICATION")
    
    # Registration
    print_subsection("1.1 User Registration")
    response = requests.post(
        f"{BASE_URL}/auth/register",
        json={"username": USERNAME, "password": PASSWORD}
    )
    
    if response.status_code == 201:
        data = response.json()
        TOKEN = data["access_token"]
        HEADERS = {"Authorization": f"Bearer {TOKEN}"}
        print(f"✓ Registered: {USERNAME}")
    else:
        print(f"✗ Failed: {response.json()}")
        return False
    
    # Login
    print_subsection("1.2 User Login")
    response = requests.post(
        f"{BASE_URL}/auth/login",
        json={"username": USERNAME, "password": PASSWORD}
    )
    
    if response.status_code == 200:
        TOKEN = response.json()["access_token"]
        HEADERS = {"Authorization": f"Bearer {TOKEN}"}
        print("✓ Login successful")
        return True
    else:
        print("✗ Login failed")
        return False

test_authentication()


  1. AUTHENTICATION


--- 1.1 User Registration ---

✓ Registered: demo_user_1771946334

--- 1.2 User Login ---

✓ Login successful


True

## 2. Market Data Subscriptions

In [21]:
async def demo_best_touch():
    print_subsection("2.1 Best Touch (Real-time Best Bid/Ask)")
    
    try:
        async with websockets.connect(WS_URL) as websocket:
            await websocket.recv()
            
            await websocket.send(json.dumps({
                "action": "subscribe",
                "data_type": "best_touch",
                "symbol": "BTCUSDT",
                "exchange": "all"
            }))
            
            update_count = 0
            while update_count < 3:
                message = await asyncio.wait_for(websocket.recv(), timeout=10.0)
                data = json.loads(message)
                
                if data.get("type") == "confirmation":
                    continue
                elif data.get("type") == "best_touch":
                    update_count += 1
                    d = data['data']
                    print(f"Update {update_count}:")
                    print(f"  Bid: ${d['bid_price']:.2f} ({d['bid_exchange']})")
                    print(f"  Ask: ${d['ask_price']:.2f} ({d['ask_exchange']})")
                    print(f"  Spread: ${d['spread']:.2f}")
                    
                    if d['spread'] < 0:
                        profit = abs(d['spread'])
                        profit_pct = (profit / d['ask_price']) * 100
                        print(f"  → ARBITRAGE: ${profit:.2f}/BTC ({profit_pct:.3f}%) - Buy {d['ask_exchange'].upper()}, Sell {d['bid_exchange'].upper()}")
                    print()
            
    except asyncio.TimeoutError:
        print("⚠ Timeout: No market data")
    except Exception as e:
        print(f"✗ Error: {e}")

await demo_best_touch()


--- 2.1 Best Touch (Real-time Best Bid/Ask) ---

Update 1:
  Bid: $63649.20 (okx)
  Ask: $63626.60 (binance)
  Spread: $-22.60
  → ARBITRAGE: $22.60/BTC (0.036%) - Buy BINANCE, Sell OKX

Update 2:
  Bid: $63649.20 (okx)
  Ask: $63626.50 (binance)
  Spread: $-22.70
  → ARBITRAGE: $22.70/BTC (0.036%) - Buy BINANCE, Sell OKX

Update 3:
  Bid: $63649.20 (okx)
  Ask: $63626.50 (binance)
  Spread: $-22.70
  → ARBITRAGE: $22.70/BTC (0.036%) - Buy BINANCE, Sell OKX



In [22]:
async def demo_trades():
    print_subsection("2.2 Trades Stream (Cross-Exchange)")
    
    try:
        async with websockets.connect(WS_URL) as websocket:
            await websocket.recv()
            
            await websocket.send(json.dumps({
                "action": "subscribe",
                "data_type": "trade",
                "symbol": "BTCUSDT",
                "exchange": "all"
            }))
            
            trade_count = 0
            binance_count = 0
            okx_count = 0
            
            while trade_count < 5:
                message = await asyncio.wait_for(websocket.recv(), timeout=10.0)
                data = json.loads(message)
                
                if data.get("type") == "confirmation":
                    continue
                elif data.get("type") == "trade":
                    t = data['data']
                    if t['price'] > 0 and t['quantity'] > 0:
                        trade_count += 1
                        if t['exchange'] == 'binance':
                            binance_count += 1
                        elif t['exchange'] == 'okx':
                            okx_count += 1
                        print(f"Trade {trade_count}: ${t['price']:.2f} x {t['quantity']} ({t['exchange'].upper()})")
            
            print(f"\n→ {binance_count} Binance, {okx_count} OKX\n")
            
    except asyncio.TimeoutError:
        print("⚠ Timeout: No trades")
    except Exception as e:
        print(f"✗ Error: {e}")

await demo_trades()


--- 2.2 Trades Stream (Cross-Exchange) ---

Trade 1: $63649.20 x 0.84 (OKX)
Trade 2: $63624.70 x 0.002 (BINANCE)
Trade 3: $63624.70 x 0.004 (BINANCE)
Trade 4: $63623.90 x 0.002 (BINANCE)
Trade 5: $63623.70 x 0.002 (BINANCE)

→ 4 Binance, 1 OKX



In [23]:
async def demo_klines():
    print_subsection("2.3 Klines (1m Candlesticks)")
    
    try:
        async with websockets.connect(WS_URL) as websocket:
            await websocket.recv()
            
            await websocket.send(json.dumps({
                "action": "subscribe",
                "data_type": "kline",
                "symbol": "SOLUSDT",
                "interval": "1m",
                "exchange": "all"
            }))
            
            kline_count = 0
            while kline_count < 1:
                message = await asyncio.wait_for(websocket.recv(), timeout=65.0)
                data = json.loads(message)
                
                if data.get("type") == "confirmation":
                    print("Waiting for 1-minute kline...")
                elif data.get("type") == "kline":
                    kline_count += 1
                    k = data['data']
                    print(f"Kline {kline_count}:")
                    print(f"  O: ${k['open']:.2f} | H: ${k['high']:.2f} | L: ${k['low']:.2f} | C: ${k['close']:.2f}")
                    print(f"  Volume: {k['volume']}\n")
            
    except asyncio.TimeoutError:
        print("⚠ Timeout: No kline (wait 60s)")
    except Exception as e:
        print(f"✗ Error: {e}")

await demo_klines()


--- 2.3 Klines (1m Candlesticks) ---

Waiting for 1-minute kline...
Kline 1:
  O: $77.23 | H: $77.24 | L: $77.06 | C: $77.16
  Volume: 10961.299999999954



In [24]:
async def demo_ewma():
    print_subsection("2.4 EWMA (Moving Average)")
    
    try:
        async with websockets.connect(WS_URL) as websocket:
            await websocket.recv()
            
            await websocket.send(json.dumps({
                "action": "subscribe",
                "data_type": "ewma",
                "symbol": "BTCUSDT",
                "exchange": "binance"
            }))
            
            ewma_count = 0
            while ewma_count < 1:
                message = await asyncio.wait_for(websocket.recv(), timeout=30.0)
                data = json.loads(message)
                
                if data.get("type") == "confirmation":
                    continue
                elif data.get("type") == "ewma":
                    ewma_count += 1
                    print(f"EWMA: ${data['data']['value']:.2f}")
            
    except asyncio.TimeoutError:
        print("⚠ Timeout: No EWMA")
    except Exception as e:
        print(f"✗ Error: {e}")

await demo_ewma()


--- 2.4 EWMA (Moving Average) ---

EWMA: $63620.00


## 3. Fund Management

In [25]:
def test_fund_management():
    print_section("3. FUND MANAGEMENT")
    
    # Deposit USDT
    print_subsection("3.1 Deposit USDT")
    response = requests.post(
        f"{BASE_URL}/deposit",
        json={"asset": "USDT", "amount": 50000.0},
        headers=HEADERS
    )
    if response.status_code == 200:
        print(f"✓ Balance: ${response.json()['new_balance']:.2f}")
    
    # Deposit BTC
    print_subsection("3.2 Deposit BTC")
    response = requests.post(
        f"{BASE_URL}/deposit",
        json={"asset": "BTC", "amount": 5.0},
        headers=HEADERS
    )
    if response.status_code == 200:
        print(f"✓ Balance: {response.json()['new_balance']:.4f} BTC")
    
    # Check balances
    print_subsection("3.3 Check Balances")
    response = requests.get(f"{BASE_URL}/balance", headers=HEADERS)
    if response.status_code == 200:
        for b in response.json()["balances"]:
            print(f"{b['asset']:6} Total: {b['total']:10.2f} | Available: {b['available']:10.2f} | Reserved: {b['reserved']:10.2f}")

test_fund_management()


  3. FUND MANAGEMENT


--- 3.1 Deposit USDT ---

✓ Balance: $50000.00

--- 3.2 Deposit BTC ---

✓ Balance: 5.0000 BTC

--- 3.3 Check Balances ---

BTC    Total:       5.00 | Available:       5.00 | Reserved:       0.00
USDT   Total:   50000.00 | Available:   50000.00 | Reserved:       0.00


## 4. Order Management

In [26]:
def test_order_management():
    global ORDER_ID_LIMIT
    
    print_section("4. ORDER MANAGEMENT")
    
    # Create limit order
    print_subsection("4.1 Create Limit Order")
    ORDER_ID_LIMIT = f"limit_buy_{int(time.time() * 1000000)}"
    
    response = requests.post(
        f"{BASE_URL}/orders",
        json={
            "token_id": ORDER_ID_LIMIT,
            "symbol": "BTCUSDT",
            "side": "buy",
            "order_type": "limit",
            "price": 50000.0,
            "quantity": 0.5
        },
        headers=HEADERS
    )
    
    if response.status_code == 201:
        order = response.json()
        print(f"✓ Order: {order['token_id'][:20]}... | Status: {order['status']} | Reserved: ${order['price'] * order['quantity']:.2f}")
    
    # Market order
    print_subsection("4.2 Create Market Order")
    time.sleep(2)
    
    response = requests.post(
        f"{BASE_URL}/orders",
        json={
            "token_id": f"market_buy_{int(time.time() * 1000000)}",
            "symbol": "BTCUSDT",
            "side": "buy",
            "order_type": "market",
            "quantity": 0.1
        },
        headers=HEADERS
    )
    
    if response.status_code == 201:
        order = response.json()
        print(f"✓ Executed at ${order['price']:.2f} | Status: {order['status']}")
    
    # IOC order
    print_subsection("4.3 Create IOC Order")
    response = requests.post(
        f"{BASE_URL}/orders",
        json={
            "token_id": f"ioc_buy_{int(time.time() * 1000000)}",
            "symbol": "BTCUSDT",
            "side": "buy",
            "order_type": "ioc",
            "price": 65000.0,
            "quantity": 0.2
        },
        headers=HEADERS
    )
    
    if response.status_code == 201:
        order = response.json()
        print(f"✓ Filled: {order.get('filled_quantity', 0)} @ ${order['price']:.2f} | Status: {order['status']}")
    else:
        print(f"✗ {response.json().get('detail', 'Error')}")
    
    # Check order
    print_subsection("4.4 Check Order Status")
    response = requests.get(f"{BASE_URL}/orders/{ORDER_ID_LIMIT}", headers=HEADERS)
    
    if response.status_code == 200:
        order = response.json()
        print(f"{order['symbol']} {order['side'].upper()} {order['order_type'].upper()} | Price: ${order['price']:.2f} | Qty: {order['quantity']} | Status: {order['status'].upper()}")
    
    return ORDER_ID_LIMIT

ORDER_ID_LIMIT = test_order_management()


  4. ORDER MANAGEMENT


--- 4.1 Create Limit Order ---

✓ Order: limit_buy_1771946336... | Status: open | Reserved: $25000.00

--- 4.2 Create Market Order ---

✓ Executed at $63627.30 | Status: filled

--- 4.3 Create IOC Order ---

✓ Filled: 0.2 @ $63628.40 | Status: filled

--- 4.4 Check Order Status ---

BTCUSDT BUY LIMIT | Price: $50000.00 | Qty: 0.5 | Status: OPEN


## 5. Order Modification (Bonus)

In [27]:
def test_order_modification(order_id):
    print_section("5. ORDER MODIFICATION (BONUS)")
    
    # Modify price
    print_subsection("5.1 Modify Price")
    response = requests.put(
        f"{BASE_URL}/orders/{order_id}",
        json={"price": 52000.0},
        headers=HEADERS
    )
    if response.status_code == 200:
        order = response.json()
        print(f"✓ New price: ${order['price']:.2f}")
    
    # Modify price and quantity
    print_subsection("5.2 Modify Price and Quantity")
    response = requests.put(
        f"{BASE_URL}/orders/{order_id}",
        json={"price": 51000.0, "quantity": 0.3},
        headers=HEADERS
    )
    if response.status_code == 200:
        order = response.json()
        print(f"✓ ${order['price']:.2f} x {order['quantity']} = ${order['price'] * order['quantity']:.2f} reserved")

test_order_modification(ORDER_ID_LIMIT)


  5. ORDER MODIFICATION (BONUS)


--- 5.1 Modify Price ---

✓ New price: $52000.00

--- 5.2 Modify Price and Quantity ---

✓ $51000.00 x 0.3 = $15300.00 reserved


## 6. Order Cancellation

In [28]:
def test_order_cancellation(order_id):
    print_section("6. ORDER CANCELLATION")
    
    response = requests.delete(f"{BASE_URL}/orders/{order_id}", headers=HEADERS)
    if response.status_code == 200:
        print(f"✓ Cancelled: {order_id[:25]}... | Funds released")

test_order_cancellation(ORDER_ID_LIMIT)


  6. ORDER CANCELLATION

✓ Cancelled: limit_buy_177194633604099... | Funds released


## 7. WebSocket Order Management (Bonus)

In [29]:
async def test_websocket_orders():
    print_section("7. WEBSOCKET ORDER MANAGEMENT (BONUS)")
    
    try:
        async with websockets.connect(f"{WS_URL}?token={TOKEN}") as websocket:
            await asyncio.wait_for(websocket.recv(), timeout=5.0)
            
            # Submit order
            ws_order_id = f"ws_order_{int(time.time() * 1000000)}"
            await websocket.send(json.dumps({
                "action": "submit_order",
                "token_id": ws_order_id,
                "symbol": "BTCUSDT",
                "side": "buy",
                "order_type": "limit",
                "price": 49000.0,
                "quantity": 0.2
            }))
            
            message = await asyncio.wait_for(websocket.recv(), timeout=5.0)
            data = json.loads(message)
            if data.get("type") == "order_submitted":
                print(f"✓ Order submitted via WebSocket: {data['order']['status']}")
            
            await asyncio.sleep(1)
            
            # Cancel order
            await websocket.send(json.dumps({
                "action": "cancel_order",
                "token_id": ws_order_id
            }))
            
            message = await asyncio.wait_for(websocket.recv(), timeout=5.0)
            data = json.loads(message)
            if data.get("type") == "order_cancelled":
                print(f"✓ Order cancelled via WebSocket")
                
    except asyncio.TimeoutError:
        print("⚠ Timeout")
    except Exception as e:
        print(f"✗ Error: {e}")

await test_websocket_orders()


  7. WEBSOCKET ORDER MANAGEMENT (BONUS)

✓ Order submitted via WebSocket: open
✓ Order cancelled via WebSocket


## Stop API Server

**Run this cell when you're done** to stop the server gracefully.

In [30]:
# Stop the server process
try:
    server_process.terminate()
    server_process.wait(timeout=5)
    print("Server stopped successfully")
except:
    try:
        server_process.kill()
        print("Server killed")
    except:
        print("Could not stop server (may have already stopped)")
        print("If server is still running, use: pkill -f 'python run_server.py'")

Server stopped successfully
